# Clasificación de textos históricos por década
## Parte 1: Machine Learning Clásico

**Tarea:** Predecir la década (primeros 3 dígitos del año) en que fue escrito un texto histórico en español.  
**Clases:** Décadas del `150` al `188` (años 1500–1889), 39 clases en total.  
**Restricción:** Únicamente modelos de `scikit-learn`, sin redes neuronales ni transformers.

## 1. Importaciones

In [ ]:
import os
import re
import unicodedata
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from typing import Any
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

SEED = 42
np.random.seed(SEED)
print('Librerías cargadas correctamente')

## 2. Carga de datos

In [ ]:
import os
from typing import Any

def load_data(filename: str, train: bool = True) -> pd.DataFrame:
    """
    Carga train.csv o eval.csv usando expresiones regulares
    para manejar correctamente textos con comas y comillas.
    """
    filepath = os.path.join(os.getcwd(), filename)
    with open(filepath, 'r', encoding='utf-8') as f:
        header = f.readline()
        raw_data: list[list[Any]] = [header.strip().split(',')]
        content = f.read()

    if train:
        regular_expression = r'"(?:[^"]|"")*",\d{3}\n'
    else:
        regular_expression = r'\d+,"(?:[^"]|"")*"\n'

    joint_data = re.findall(regular_expression, content, flags=re.MULTILINE)
    for data in joint_data:
        data = data.strip().replace('\n', '')
        if train:
            text = data[:-4]
        else:
            text = data[data.find(',') + 1:]
        text = text[1:-1].strip()
        if train:
            decade = data[-3:]
            raw_data.append([text, int(decade)])
        else:
            id_ = data[:data.find(',')]
            raw_data.append([int(id_), text])

    df = pd.DataFrame(raw_data[1:], columns=raw_data[0])
    return df

train_df = load_data('data/train.csv', train=True)
eval_df  = load_data('data/eval.csv',  train=False)

print(f'Train: {len(train_df)} registros')
print(f'Eval:  {len(eval_df)} registros')
train_df.head(3)

## 3. Análisis exploratorio

In [ ]:
vc = train_df['decade'].value_counts().sort_index()
print(f'Décadas únicas : {train_df["decade"].nunique()}')
print(f'Rango          : {train_df["decade"].min()} – {train_df["decade"].max()}')
print(f'Ejemplos/clase : min={vc.min()}, max={vc.max()}, media={vc.mean():.0f}')

plt.figure(figsize=(14, 3))
plt.bar(vc.index, vc.values, color='steelblue')
plt.xlabel('Década')
plt.ylabel('Ejemplos')
plt.title('Distribución de ejemplos por década')
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de longitud de textos
lens = train_df['text'].str.len()
print(lens.describe())

plt.figure(figsize=(10, 3))
plt.hist(lens, bins=60, color='steelblue', edgecolor='white')
plt.xlabel('Longitud del texto (caracteres)')
plt.title('Distribución de longitud de textos')
plt.tight_layout()
plt.show()

## 4. Preprocesamiento

Se aplica una limpieza mínima para preservar señales ortográficas históricas:
- Colapso de saltos de línea y espacios múltiples
- Conversión a minúsculas
- Unión de palabras separadas por guiones (artefacto OCR)
- Eliminación de caracteres no alfabéticos excepto puntuación básica

Se conserva la ortografía histórica (uso de 'v/b', grafías medievales, etc.) porque es señal clave para distinguir épocas.

In [ ]:
def preprocess(text: str) -> str:
    """
    Limpieza mínima que preserva características ortográficas históricas.
    Estas variaciones son señal clave para la clasificación por época.
    """
    if not isinstance(text, str):
        return ''
    # Colapsar saltos de línea y tabulaciones
    text = re.sub(r'[\r\n\t]+', ' ', text)
    # Minúsculas
    text = text.lower()
    # Unir palabras separadas por guiones (artefacto OCR frecuente)
    text = re.sub(r'(\w+)([-¬>])\s+(\w+)', r'\1\3', text)
    # Eliminar caracteres no alfabéticos (conserva letras con tilde y ñ)
    text = re.sub(r'[^a-záéíóúüñ\s\d,.:;\-\'*¿?]', '', text)
    # Colapsar espacios múltiples
    return re.sub(r' +', ' ', text).strip()

train_df['text_clean'] = train_df['text'].apply(preprocess)
eval_df['text_clean']  = eval_df['text'].apply(preprocess)

# Verificación
idx = 0
print('Original:')
print(repr(train_df['text'].iloc[idx][:200]))
print('\nLimpio:')
print(repr(train_df['text_clean'].iloc[idx][:200]))

In [ ]:
X_train = train_df['text_clean'].values
y_train = train_df['decade'].values
X_eval  = eval_df['text_clean'].values

print(f'Train: {len(X_train)} ejemplos | Eval: {len(X_eval)} ejemplos')
print(f'Clases únicas: {len(np.unique(y_train))}')

## 5. Construcción del modelo

El pipeline combina dos representaciones TF-IDF:

| Componente | Descripción | Razón |
|---|---|---|
| `TfidfVectorizer(analyzer='char', ngram_range=(3,5))` | N-gramas de caracteres que cruzan límites de palabra | Captura patrones ortográficos de época, robusto a ruido OCR |
| `TfidfVectorizer(analyzer='word', ngram_range=(1,2))` | Unigramas y bigramas de palabras | Captura vocabulario y frases características de cada siglo |
| `FeatureUnion` con pesos `{char:1.0, word:0.3}` | Combinación ponderada | Da más importancia a la morfología que al vocabulario |
| `LogisticRegression(C=10, class_weight='balanced')` | Clasificador lineal multinomial | Eficiente con vocabularios grandes, maneja clases balanceadas |

La clave del modelo es usar `analyzer='char'` (sin word-boundary) en lugar de `char_wb`, ya que captura n-gramas que cruzan los límites de palabras — por ejemplo `" de"`, `"ón "` — que son altamente característicos de cada época histórica.

In [ ]:
def build_pipeline() -> Pipeline:
    """
    Construye el pipeline final con FeatureUnion de dos TF-IDF
    y LogisticRegression con pesos balanceados.
    """
    char_vec = TfidfVectorizer(
        analyzer='char',        
        ngram_range=(3, 5),
        sublinear_tf=True,
        min_df=2,
    )
    word_vec = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        sublinear_tf=True,
        max_features=100_000,
    )
    transformer_list: Any = [('char', char_vec), ('word', word_vec)]
    features = FeatureUnion(
        transformer_list=transformer_list,
        transformer_weights={'char': 1.0, 'word': 0.3},
    )
    return Pipeline([
        ('features', features),
        ('clf', LogisticRegression(
            C=10.0,
            solver='saga',
            max_iter=1000,
            class_weight='balanced',
            random_state=SEED,
        )),
    ])

pipeline = build_pipeline()
print('Pipeline construido:')
print(pipeline)

## 6. Validación con Cross-Validation

Se usa validación cruzada estratificada de 5 folds para estimar el accuracy del modelo de forma robusta.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print('Corriendo validación cruzada (5 folds)...')
cv_scores = cross_val_score(
    pipeline, X_train, y_train,
    cv=cv, scoring='accuracy', n_jobs=1, verbose=1,
)

print(f'\nCV Accuracy por fold: {cv_scores}')
print(f'CV Accuracy media  : {cv_scores.mean():.4f}')
print(f'CV Accuracy std    : {cv_scores.std():.4f}')
print(f'Baseline aleatorio : {1/39:.4f}')

## 7. Entrenamiento del modelo final

Se reentrena el pipeline con **todos los datos de entrenamiento** para maximizar la información disponible antes de predecir sobre eval.

In [ ]:
print('Entrenando modelo final sobre todos los datos...')
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, pipeline.predict(X_train))
print(f'Train accuracy (referencia): {train_acc:.4f}')
print('Modelo entrenado correctamente.')

## 8. Análisis de errores

In [ ]:
# Usar el último fold de CV para análisis de errores
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train,
)

pipe_val = build_pipeline()
pipe_val.fit(X_tr, y_tr)
val_pred = pipe_val.predict(X_val)

print(f'Val accuracy (hold-out 15%): {accuracy_score(y_val, val_pred):.4f}')

# Distribución del error en décadas
errors = np.abs(y_val.astype(int) - val_pred.astype(int))
print('\nDistribución del error (décadas de diferencia):')
for k in range(6):
    pct = (errors == k).mean() * 100
    print(f'  |error| = {k}:  {pct:.1f}%')
print(f'  |error| >= 6:  {(errors >= 6).mean()*100:.1f}%')

plt.figure(figsize=(10, 3))
plt.hist(errors, bins=range(0, 40), color='steelblue', edgecolor='white')
plt.xlabel('|Década real − Década predicha|')
plt.ylabel('Ejemplos')
plt.title('Distribución del error en décadas')
plt.tight_layout()
plt.show()

## 9. Guardar el modelo

In [ ]:
joblib.dump(pipeline, 'modelo_parte1.joblib')
print('Modelo guardado en: modelo_parte1.joblib')

# Verificación de carga
loaded = joblib.load('modelo_parte1.joblib')
check = loaded.predict(X_train[:5])
print(f'Predicciones (primeros 5): {check}')
print(f'Etiquetas reales:          {y_train[:5]}')

## 10. Generación del archivo de respuesta

In [ ]:
eval_pred = pipeline.predict(X_eval)

submission = pd.DataFrame({
    'id':     eval_df['id'],
    'answer': eval_pred,
})

submission.to_csv('submission.csv', index=False)
print(f'submission.csv generado: {len(submission)} predicciones')
print(f'Rango de décadas predichas : {eval_pred.min()} – {eval_pred.max()}')
print(f'Décadas únicas predichas   : {len(np.unique(eval_pred))}')
print()
print(submission.head(10).to_string(index=False))

In [ ]:
# Distribución de predicciones vs entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

vc_train = pd.Series(y_train).value_counts().sort_index()
vc_pred  = pd.Series(eval_pred).value_counts().sort_index()

axes[0].bar(vc_train.index, vc_train.values, color='steelblue')
axes[0].set_title('Distribución train')
axes[0].set_xlabel('Década')

axes[1].bar(vc_pred.index, vc_pred.values, color='coral')
axes[1].set_title('Predicciones eval')
axes[1].set_xlabel('Década')

plt.tight_layout()
plt.show()

## Resumen del modelo

| Componente | Detalle |
|---|---|
| **Preprocesamiento** | Limpieza mínima: minúsculas, colapso de saltos de línea, unión de palabras con guión, eliminación de caracteres no alfabéticos |
| **Features** | `FeatureUnion` de TF-IDF char (3-5 gramas, cruza límites de palabra) + TF-IDF word (1-2 gramas), pesos `{char:1.0, word:0.3}` |
| **Modelo** | `LogisticRegression(C=10, solver='saga', class_weight='balanced')` |
| **Validación** | Cross-validation estratificada 5-fold |
| **Librería** | `scikit-learn` exclusivamente |
| **Modelo guardado** | `modelo_parte1.joblib` |
| **Submission** | `submission.csv` (formato `id,answer`) |